In [1]:

import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader , TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
df_train = pd.read_csv("../data/mnist_train.csv")
df_test = pd.read_csv("../data/mnist_test.csv")


In [4]:
torch.backends.cudnn.benchmark = True

In [5]:
df_train.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
y_train = df_train.iloc[:,0]
X_train = df_train.iloc[:,1:]


In [7]:
y_train.head()

0    5
1    0
2    4
3    1
4    9
Name: label, dtype: int64

In [8]:
X_train.head()

,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,1x10,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
X_train = X_train.values.astype("float32") / 255.0
y_train = y_train.values.astype("int64")

In [10]:
X_train , X_val , y_train , y_val = train_test_split(X_train, y_train,test_size=.1 ,shuffle= True, stratify= y_train , random_state= 43)

In [11]:
X_train = (X_train - 0.1307) / 0.3081
X_val = (X_val - 0.1307) / 0.3081

In [12]:
X_train = X_train.reshape(-1,784,1)
X_val = X_val.reshape(-1,784,1)

In [13]:
train_set = TensorDataset(torch.from_numpy(X_train),torch.from_numpy(y_train))
val_set = TensorDataset(torch.from_numpy(X_val),torch.from_numpy(y_val))

In [14]:
train_loader = DataLoader(
    train_set,
    batch_size= 128,
    shuffle=True,
    num_workers=4,
    pin_memory= True
)
val_loader = DataLoader(
    train_set,batch_size= 256,
    shuffle=False,
    num_workers= 4,
    pin_memory= 4
)


In [15]:
class VanillaRnn(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.RNN(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 10)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [16]:
class Lstm(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 10)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [17]:
class Gru(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.GRU(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 10)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [18]:
model_rnn = VanillaRnn(1,64,2).to(device)
model_lstm = Lstm(1,64,2).to(device)
model_gru = Gru(1,64,2).to(device)

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer_rnn = torch.optim.AdamW(model_rnn.parameters(),lr = 0.0001, weight_decay=1e-4)
optimizer_lstm = torch.optim.AdamW(model_lstm.parameters(),lr = 0.0001, weight_decay=1e-4)


In [20]:
optimizer_gru = torch.optim.AdamW(model_gru.parameters(),lr = 0.0001, weight_decay=1e-4)

In [21]:
class Trainer:
    def __init__(self, model, criterion, optimizer, device, save_path="best_rnn.pt"):
        self.model = model.to(device)
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.save_path = save_path
        self.best_val_loss = float("inf")

    def _run_epoch(self, loader, train):
        self.model.train(train)  # train=True -> train mode, False -> eval mode
        total_loss, correct, total = 0.0, 0, 0

        with torch.set_grad_enabled(train):
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)

                out = self.model(x)
                loss = self.criterion(out, y)

                if train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()

                total_loss += loss.item() * x.size(0)
                correct += (out.argmax(dim=1) == y).sum().item()
                total += x.size(0)

        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader, epochs):
        for epoch in range(epochs):
            train_loss, train_acc = self._run_epoch(train_loader, train=True)
            val_loss, val_acc = self._run_epoch(val_loader, train=False)

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.save_path)
                print(f"save!!! Epoch {epoch:02d}/{epochs} | val CELoss: {val_loss:.4f} | val acc: {val_acc:.4f}")

            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:02d}/{epochs} | train CELoss: {train_loss:.4f} acc: {train_acc:.4f} "
                      f"| val CELoss: {val_loss:.4f} acc: {val_acc:.4f}")
        return self

In [ ]:
trainer = Trainer(
    model=model_rnn,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_rnn,
    device=device,
    save_path="best_rnn.pt",
)
trainer.fit(train_loader, val_loader, epochs=40)

In [ ]:
trainer = Trainer(
    model=model_lstm,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_lstm,
    device=device,
    save_path="best_lstm.pt",
)
trainer.fit(train_loader, val_loader, epochs=40)

In [ ]:
trainer = Trainer(
    model=model_gru,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_gru,
    device=device,
    save_path="best_gru.pt",
)
trainer.fit(train_loader, val_loader, epochs=40)